In [48]:
!pip install pyserial

In [3]:
import serial, time
#!pip install pyserial

In [4]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [5]:
print(serial)

<module 'serial' from 'C:\\Users\\caleb\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [6]:
print(serial.__file__)

C:\Users\caleb\anaconda3\Lib\site-packages\serial\__init__.py


In [7]:
print(serial.__version__)

3.5


In [8]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [9]:
baudrate = 115200

In [10]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM3'#windows

In [11]:
ser = serial.Serial(portname, baudrate, timeout=5)

SerialException: could not open port 'COM3': FileNotFoundError(2, 'The system cannot find the file specified.', None, 2)

In [12]:
ser.in_waiting

NameError: name 'ser' is not defined

In [13]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [14]:
read_all(ser)

NameError: name 'ser' is not defined

In [15]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [16]:
read_one_line(ser)

NameError: name 'ser' is not defined

In [17]:
read_all(ser)

NameError: name 'ser' is not defined

In [18]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [19]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

# Break an integer into two bytes

In [20]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [21]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [48]:
def GrabberAngle(L1, L2, X, Z):
    Theta_intermediate1 = 90 - rtd * np.arctan2(Z, X)
    
    Theta_intermediate3 = rtd*np.arccos(((X**2 + Z**2) + L2**2 - L1**2)/(2*(X**2 + Z**2)**.5 * L2 ))
    Angle_Between = Theta_intermediate3 + Theta_intermediate1
    return Angle_Between

In [68]:
def XZLocation(l1, l2, X, Z):
    #Define Link lengths

    
    #distance from origin to tip
    r_squared = X**2 + Z**2
    
    #law of cos for angle between links
    alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
    
    alpha = np.arccos(alpha_temp)
    
    
    
    #vertical angle theorem for theta 2
    theta2 = 180 - alpha*rtd
    
    #triangle in link1 co-ordinant system for psi
    psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))*rtd
    
    #angle of r to x-axis
    beta = np.arctan2(Z,X)*rtd
    
    #difference in beta and psi is theta 1
    theta1 = beta - psi

    return theta1, theta2
    

In [71]:
def theta1interpolation(theta1):
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 1000    # minimum servo value
    max_new = 2000    # maximum servo value

    #linear interpolate for the first theta value
    myint = min_new + ((theta1-theta_min)*(max_new-min_new))/(theta_max-theta_min)
    return myint

np.float64(83.10789742065361)

In [ ]:
def theta2interpolation(theta2):
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 1000    # minimum servo value
    max_new = 2000    # maximum servo value
    #linear interpolate for the second theta value; 180 and 90 included to account for robots home position as The servo has 1000 -> theta2=90, 2000 -> theta2 = -90
    myint2 = min_new + (((180-(90+theta2))-theta_min)*(max_new-min_new))/(theta_max-theta_min)
    print('\n',myint2)
    return myint2

In [56]:
L1 = 25
L2 = 25

ObjectPickupX = -7
ObjectPickupY = 2

ObstacleX = 3
ObstacleY = 9

DropOffX = 9
DropOffY = 9

Height1 = 5 # Hover Height
Height2 = 7 #Grab Height
CL = 2 #Obstacle Clearance

Theta1 = 0
X1 = 0
Z1 = 0

Theta2 = 0
X2 = 0
Z2 = 0

Theta3 = 0
X3 = 0
Z3 = 0

Theta4 = 0
X4 = 0
Z4 = 0

Theta5 = 0
X5 = 0
Z5 = 0

Theta6 = 0
X6 = 0
Z6 = 0

Theta1 = 0
X7 = 0
Z7 = 0



In [59]:
#Step 1 - Go to correct angle to pick up object
Theta1 = rtd*np.arctan2(ObjectPickupY, ObjectPickupX)

In [61]:
#Step 2 - Go to correct height and radius to pick up object
Theta2 = Theta1
X2 = (ObjectPickupY**2 + ObjectPickupX**2)**.5
Z2 = Height1
Link1_2, Link2_2 = XZLocation(L1, L2, X2, Z2)


In [ ]:
#Step 3 - Grab Object
Theta3 = Theta2
X3 = X2
Z3 = Height2
Link1_3, Link2_3 = XZLocation(L1, L2, X3, Z3)

In [63]:
#Step 4 - Go to clearance distance to be able to move around obstacle
Theta4 = Theta3
X4 = (ObstacleY**2 + ObstacleX**2)**.5 - CL
Z4 = Height1
Link1_4, Link2_4 = XZLocation(L1, L2, X4, Z4)

In [ ]:
#Step 5 - Go to correct angle to drop off object
Theta5 =  rtd * np.arctan2(DropOffY, DropOffX)
X5 = X4
Z5 = Z4
Link1_5, Link2_5 = XZLocation(L1, L2, X5, Z5)

In [ ]:
#Step 6 - Go to correct location to drop off object
Theta6 = Theta5
X6 = (DropOffY**2 + DropOffX**2)**.5
Z6 = Z5
Link1_6, Link2_6 = XZLocation(L1, L2, X6, Z6)

In [ ]:
#Step 7 - Drop Off object
Theta7 = Theta6
X7 = X6
Z7 = Height2
Link1_7, Link2_7 = XZLocation(L1, L2, X7, Z7)

In [70]:
myint1 = 1700
byte1, byte2 = break_into_two(myint1)

myint2 = 1600
byte3, byte4 = break_into_two(myint2)

myint3 = 1500
byte5, byte6 = break_into_two(myint3)

myint4 = 1400
byte7, byte8 = break_into_two(myint4)

myint5 = 1300
byte9, byte10 = break_into_two(myint5)

In [71]:
WriteByte(ser, int(byte1))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")


 servo1=My int: 1700  
 servo1=My int2: 1600  
 servo1=My int3: 1500  
 servo1=My int4: 1400  
 servo1=My int5: 1300  


In [47]:
ser.close()